In [42]:
#imports
import pandas as pd
import csv
import os
from datasets import Dataset, DatasetDict
from datasets import Features, Sequence, Value, ClassLabel
import itertools

In [2]:
#read all files into one large dataframe
cwd = os.getcwd()

input_path = "../sbb_ner_data/zefys/"
files = [file for file in os.listdir(input_path) if file.endswith('.tsv')]

/mnt/c/Users/b-ss117/Work Folders/Eigene Dateien/dev/sbb_ner_update


In [27]:
#read TOKEN and NE-TAG over all files as one list, split into sentences by using No. col

#def lists to add all sentences to
tokens=[]
tags=[]

for file in files:
    #read input as pd df
    input = pd.read_csv(input_path + filename, sep="\t", comment="#", quoting=csv.QUOTE_NONE)

    #find sentence beginnings
    sent_starts = input.loc[input["No."] == 0]
    sent_starts_idx = sent_starts.index.tolist()

    #read into nested lists based on sentence structure
    for i, index in enumerate(sent_starts_idx):
        if index != 0:
        
            sent_tokens = input["TOKEN"][sent_starts_idx[i-1]:index-1].tolist()
            sent_tags = input["NE-TAG"][sent_starts_idx[i-1]:index-1].tolist()
    
            tokens.append(sent_tokens)
            tags.append(sent_tags)

print(len(tokens), len(tags))

10200 10200


In [ ]:
#TO DO: check NE-EMB usage (multilabel token classification?)
#https://discuss.huggingface.co/t/multi-label-token-classification/16509/5

In [33]:
# create HF dataset from lists

#create new, minimal df based on lists
df_dataset = pd.DataFrame(
{'id': range(len(tokens)),
'tokens': tokens,
'ner_tags': tags
})

#def label_lists
tags_flattened = list(itertools.chain(*tags))
label_list = list(set(tags_flattened))

#transform data into Datasets format and set labels for ClassLabel based on label_list
zefys_dataset = Dataset.from_pandas(df_dataset)
zefys_dataset = zefys_dataset.cast_column("ner_tags", Sequence(ClassLabel(names=label_list)))

Casting the dataset: 100%|███████████████████████████████████████████████| 10200/10200 [00:03<00:00, 2588.98 examples/s]


In [36]:
zefys_dataset.features
#zefys_dataset[0]

{'id': Value(dtype='int64', id=None),
 'tokens': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
 'ner_tags': Sequence(feature=ClassLabel(names=['I-PER', 'B-PER', 'O', 'B-LOC', 'I-LOC', 'B-ORG', 'I-ORG'], id=None), length=-1, id=None)}

In [54]:
# add train/test/val split to dataset
# https://discuss.huggingface.co/t/how-to-split-main-dataset-into-train-dev-test-as-datasetdict/1090/2

# 90% train, 10% test + validation
train_testvalid = zefys_dataset.train_test_split(test_size=0.1)
# Split the 10% test + valid in half test, half valid
test_valid = train_testvalid['test'].train_test_split(test_size=0.5)
# gather everyone if you want to have a single DatasetDict
zefys_dataset_splits = DatasetDict({
    'train': train_testvalid['train'],
    'test': test_valid['test'],
    'valid': test_valid['train']})

#zefys_dataset_splits["test"][3]
zefys_dataset_splits

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 9180
    })
    test: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 510
    })
    valid: Dataset({
        features: ['id', 'tokens', 'ner_tags'],
        num_rows: 510
    })
})

In [53]:
#https://stackoverflow.com/a/72021864
zefys_dataset_splits.save_to_disk("data/zefys2025.hf")

Saving the dataset (1/1 shards): 100%|██████████████████████████████████████| 510/510 [00:00<00:00, 31207.16 examples/s]
